# 01 Data Collection and Cleaning

This notebook loads the raw job posting dataset, checks its structure, cleans inconsistent values, validates missing data, and saves a processed version for analysis.

Project: SkillMap — UAE–Canada Tech Job Market Intelligence Dashboard

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
RAW_DATA_PATH = Path("../data/raw/job_postings_sample.csv")
PROCESSED_DATA_PATH = Path("../data/processed/cleaned_jobs.csv")

In [ ]:
df = pd.read_csv(RAW_DATA_PATH)

df.head()

In [ ]:
df.shape

In [ ]:
df.columns.tolist()

In [ ]:
df.info()

In [ ]:
missing_summary = df.isna().sum().sort_values(ascending=False)
missing_summary

In [ ]:
clean_df = df.copy()

In [ ]:
text_columns = [
    "job_id",
    "country",
    "city",
    "job_title",
    "company",
    "source",
    "job_url",
    "employment_type",
    "work_mode",
    "seniority",
    "currency",
    "education_requirement",
    "description_short",
    "skills_text",
    "notes"
]

for col in text_columns:
    clean_df[col] = clean_df[col].astype("string").str.strip()

In [ ]:
clean_df["collection_date"] = pd.to_datetime(clean_df["collection_date"], errors="coerce")

In [ ]:
numeric_columns = [
    "salary_min",
    "salary_max",
    "years_experience_min",
    "years_experience_max"
]

for col in numeric_columns:
    clean_df[col] = pd.to_numeric(clean_df[col], errors="coerce")

In [ ]:
clean_df["country"] = clean_df["country"].replace({
    "UAE": "UAE",
    "United Arab Emirates": "UAE",
    "Canada": "Canada"
})

In [ ]:
clean_df["country"].value_counts()

In [ ]:
clean_df["employment_type"] = clean_df["employment_type"].replace({
    "Full Time": "Full-time",
    "Full time": "Full-time",
    "Permanent employment Full time": "Full-time",
    "Full-time Internship": "Internship",
    "Full-time Internship/Co-op": "Internship/Co-op",
    "Full-time Internship / Co-op": "Internship/Co-op",
    "Internship / Co-op": "Internship/Co-op"
})

In [ ]:
clean_df["employment_type"].value_counts(dropna=False)

In [ ]:
clean_df["work_mode"] = clean_df["work_mode"].replace({
    "On site": "On-site",
    "Onsite": "On-site",
    "In person": "On-site",
    "In-person": "On-site",
    "Remote": "Remote",
    "Hybrid": "Hybrid"
})

In [ ]:
clean_df["work_mode"].value_counts(dropna=False)

In [ ]:
clean_df["seniority"] = clean_df["seniority"].replace({
    "Intern": "Internship",
    "Entry level": "Entry-level",
    "Early-career": "Junior",
    "Coop": "Co-op",
    "Co-op": "Co-op",
    "Associate": "Associate",
    "Junior": "Junior",
    "Internship": "Internship"
})

In [ ]:
clean_df["seniority"].value_counts(dropna=False)

In [ ]:
def detect_salary_period(notes):
    if pd.isna(notes):
        return pd.NA
    
    notes_lower = str(notes).lower()
    
    if "hourly" in notes_lower:
        return "Hourly"
    if "annually" in notes_lower or "annual" in notes_lower:
        return "Annual"
    if "monthly" in notes_lower:
        return "Monthly"
    
    return pd.NA

clean_df["salary_period"] = clean_df["notes"].apply(detect_salary_period)

In [ ]:
clean_df[["job_id", "salary_min", "salary_max", "currency", "salary_period", "notes"]]

In [ ]:
clean_df["has_salary"] = clean_df["salary_min"].notna() | clean_df["salary_max"].notna()

In [ ]:
def categorize_experience(min_years):
    if pd.isna(min_years):
        return "Not specified"
    if min_years == 0:
        return "0 years / student-friendly"
    if min_years <= 1:
        return "1 year"
    if min_years <= 2:
        return "2 years"
    if min_years <= 3:
        return "3 years"
    return "4+ years"

clean_df["experience_category"] = clean_df["years_experience_min"].apply(categorize_experience)

In [ ]:
clean_df["experience_category"].value_counts()

In [ ]:
clean_df["entry_level_with_high_experience"] = (
    clean_df["seniority"].isin(["Entry-level", "Junior", "Associate"]) &
    (clean_df["years_experience_min"] >= 2)
)

In [ ]:
clean_df[clean_df["entry_level_with_high_experience"] == True][
    ["job_id", "job_title", "company", "seniority", "years_experience_min", "years_experience_max", "notes"]
]

In [ ]:
clean_df.to_csv(PROCESSED_DATA_PATH, index=False)

print(f"Cleaned jobs saved to: {PROCESSED_DATA_PATH}")
print(f"Rows saved: {len(clean_df)}")
print(f"Columns saved: {len(clean_df.columns)}")

In [ ]:
test_cleaned = pd.read_csv(PROCESSED_DATA_PATH)

print("Cleaned jobs shape:", test_cleaned.shape)

test_cleaned.head()

## Phase 1 Output

This notebook cleans the raw job-posting dataset and saves the cleaned job-level table to:

- `data/processed/cleaned_jobs.csv`

Skill extraction is handled separately in `02_skill_extraction.ipynb`, which converts the `skills_text` column into a normalized job-skills table.